# FSCT 8561 — Lab 5: Automated Web Vulnerability Analysis Using Python
**Web Application Security | 3-hour guided lab**  
**Instructor:** Dr. Maryam R. Aliabadi

## Your mission
You are an application security analyst evaluating a **local training application**, OWASP Juice Shop. First you will learn and practice individual Python techniques. **Only after completing the tutorials** will you design two separate scripts yourself:

1. `http_scanner.py` — collects and logs HTTP observations.
2. `vulnerability_detector.py` — reviews responses for potential security indicators and summarizes findings.

**Important:** The tutorial cells are examples, not the final scripts. The final two scripts are student-developed deliverables. Use only your own local Juice Shop (`http://localhost:3000`).

## Learning outcomes
By the end of the lab, you should be able to:
- Explain GET/POST, query parameters, status codes, response bodies and headers.
- Use Python `requests` to contact a known web endpoint and inspect its response.
- Use variables, lists, `for` loops, `if` statements, dictionaries and functions to organize observations.
- Distinguish **potential indicators** from confirmed XSS or SQL injection vulnerabilities.
- Develop two short Python programs and document their results.



## Environment setup and authorized scope
We use **OWASP Juice Shop**, a deliberately vulnerable training application. It must run on **your own machine**. The exercises use `localhost`, which means *this computer*. Never change the target to an external website for this lab.

### Launch Juice Shop

```bash
sudo apt update
sudo apt install -y docker.io
sudo systemctl enable docker --now
sudo docker pull bkimminich/juice-shop
sudo docker run --rm -d --name fsct8561-juice -p 127.0.0.1:3000:3000 bkimminich/juice-shop
```

Visit **http://127.0.0.1:3000** in your browser. You should see the Juice Shop storefront.

**If Jupyter runs in a different machine/container**, `localhost` will refer to *that* environment. Run Jupyter on the same machine as Docker for this lab.

**Stop after class:** `sudo docker stop fsct8561-juice`.

 Install the Python dependency in your notebook environment if necessary:

```python
%pip install requests
```

**Scope:** Test only your own training instance. Do not scan public websites or classmates' computers. Juice Shop is deliberately vulnerable; observed behavior can vary with version and endpoint.

# Part A — Understanding HTTP before writing a scanner

A browser is an HTTP client. It sends a **request** to a server; the server sends a **response**. A request has a method (`GET` or `POST`), a URL, optional parameters, headers, and sometimes a body. A response includes a status code, headers and a body.

For example, a search request might look like:

`GET /rest/products/search?q=apple`

The `q` parameter contains the search term. The server decides how to process it. A `200` means the HTTP request succeeded, **not** that the application is secure. A `404` means the path was not found; a `405` means the method is not allowed; a `500` indicates a server-side error.

### A1. Your first request
`requests.get(url, timeout=5)` sends a GET request. The `response` variable holds the result. `timeout=5` prevents Python from waiting indefinitely.

In [ ]:
import requests

base_url = "http://localhost:3000"
response = requests.get(base_url, timeout=5)

print("HTTP status:", response.status_code)
print("Response length:", len(response.content), "bytes")

**Check your understanding:** What is the difference between `response.status_code` and `len(response.content)`? Write your answer below.

**Answer:** ...

### A2. Reading response headers and content
`response.headers` acts like a dictionary of header names and values. `.get("Content-Type")` returns the header value if it exists. `response.text[:100]` shows just the first 100 characters, rather than printing an entire web page.

In [ ]:
print("Content type:", response.headers.get("Content-Type"))
print("First 100 characters:", response.text[:100])

**Try it:** Change `100` to `50` and compare the output. Explain what slicing (`[:50]`) does.

### A3. Sending a search parameter
The `params` argument builds the query string safely. The next request searches for the word `apple`. Notice that `requests` handles URL encoding for you.

In [ ]:
search_url = base_url + "/rest/products/search"
r = requests.get(search_url, params={"q": "apple"}, timeout=5)

print("Requested URL:", r.url)
print("Status:", r.status_code)
print("Content type:", r.headers.get("Content-Type"))
print("First 150 characters:", r.text[:150])

**Practice A:** Run the example again using the search word `banana`. Which part of the request changed? Did the status code change? Do not assume that a different response length indicates a vulnerability.

### A4. GET versus POST
GET typically retrieves information; POST typically submits data. The HTTP method alone does not determine whether a request is secure. In this demonstration, POST is sent to the search path **only to observe how the server handles an unsupported method**. A 404 or 405 is an ordinary outcome.

In [ ]:
post_response = requests.post(
    search_url,
    data={"q": "apple"},
    timeout=5
)
print("POST status:", post_response.status_code)
print("POST response length:", len(post_response.content))

**Practice B:** In one or two sentences, compare the GET and POST results. Why does a `405` response not necessarily indicate a vulnerability?

# Part B — Python building blocks for a scanner

Your final HTTP scanner will repeat requests and save observations. Before writing it, practice four familiar Python tools: **lists, loops, dictionaries and functions**. The examples below are intentionally small.

### B1. Lists and loops
A list stores several items. A `for` loop visits them one by one. This is how a scanner can test multiple inputs without duplicating code.

In [ ]:
search_terms = ["apple", "banana", "orange"]

for term in search_terms:
    print("Search term:", term)

**Practice C:** Add `lemon` to the list and run the cell again. How many times does the loop run?

### B2. Dictionaries: one record per request
A dictionary groups named values. For example, an HTTP observation can include `method`, `endpoint`, `status_code` and `response_length`. This is easier to interpret than a list of unnamed values.

In [ ]:
observation = {
    "method": "GET",
    "endpoint": "/rest/products/search",
    "status_code": r.status_code,
    "response_length": len(r.content)
}

print(observation)
print("Status:", observation["status_code"])

**Practice D:** Add a `content_type` key to `observation` using `r.headers.get("Content-Type")`. Print that value.

### B3. Combining a loop and a list of records
`append()` adds a new dictionary to a list. Notice that this example uses only two search terms and records just two fields; **it is not the complete scanner**.

In [ ]:
records = []

for term in ["apple", "banana"]:
    result = requests.get(search_url, params={"q": term}, timeout=5)
    records.append({"input": term, "status_code": result.status_code})

print(records)

**Practice E:** Add `response_length` to each dictionary. Explain why a list of dictionaries is helpful for generating a report.

### B4. Writing a simple function
A function gives a repeated task a name. Parameters are inputs to the function, and `return` sends a value back. This example returns only the HTTP status code, leaving the full scanner design to you.

In [ ]:
def get_status(search_word):
    result = requests.get(search_url, params={"q": search_word}, timeout=5)
    return result.status_code

print(get_status("apple"))

**Practice F:** Call `get_status("banana")`. Then explain what `search_word` means inside the function. You will decide later whether your scanner needs a function that returns a whole dictionary instead of a single number.

### B5. Handling network errors
If the local server is stopped, a request can fail before an HTTP response exists. `try`/`except` prevents a script from crashing and lets you report the connection problem. A connection error is **not** proof of a vulnerability.

In [ ]:
try:
    test_response = requests.get(base_url, timeout=5)
    print("Connected. Status:", test_response.status_code)
except requests.RequestException as error:
    print("Could not connect:", error)

**Practice G:** Explain why a scanner should include a timeout and handle `requests.RequestException`.

# Part C — Recognizing potential web security indicators

In this part, you will learn the **individual checks** needed by your vulnerability detector. Do not claim a confirmed vulnerability solely because a string appears in a response or because the server returns an error. Manual verification is essential.

### C1. Reflected input and XSS
Cross-site scripting (XSS) occurs when untrusted data is treated as executable browser content in an unsafe context. A basic scanner can check whether submitted text is reflected in an HTTP response. **Reflection alone does not establish XSS:** the browser may escape the text, return it as JSON, or never execute it.

We start with a harmless marker rather than executable JavaScript.

In [ ]:
marker = "fsct8561-marker-123"
marker_response = requests.get(search_url, params={"q": marker}, timeout=5)

if marker in marker_response.text:
    print("Marker appears in the response — investigate its context")
else:
    print("Marker was not found in this response")

**Practice H:** Test the non-executable HTML string `<u>fsct8561-test</u>` as a search term. Does it appear unchanged? If so, does that prove XSS? Explain why or why not. Keep the tests limited to the local lab instance.

### C2. SQL injection indicators
SQL injection can occur when an application builds database queries using untrusted input. A beginner-level check compares ordinary input with **simple syntax probes** and looks for unusual server errors. An HTTP 500 response or a database-related error is an **indicator**, not proof of SQL injection or successful exploitation.

In [ ]:
normal_input = "apple"
probe_input = "'"

normal = requests.get(search_url, params={"q": normal_input}, timeout=5)
probe = requests.get(search_url, params={"q": probe_input}, timeout=5)

print("Normal input status:", normal.status_code)
print("Probe input status:", probe.status_code)

**Practice I:** Explain what you observed. Why is comparing the probe response to a normal baseline more informative than looking at the probe alone?

### C3. Checking for database error messages
A program can look for known error phrases in a response. `.lower()` makes the comparison case-insensitive. This example checks **one** phrase to demonstrate the technique.

In [ ]:
error_phrase = "SQLITE_ERROR"

if error_phrase.lower() in probe.text.lower():
    print("Database-related error text found; manual review required")
else:
    print("No matching database error text in this response")

**Practice J:** Suggest two additional database-error phrases that a detector might check. Explain why a matching phrase alone cannot establish successful SQL injection.

### C4. Security response headers
Security headers help browsers apply restrictions. `Content-Security-Policy` can limit where executable content comes from; `X-Content-Type-Options: nosniff` helps prevent MIME-type sniffing. Missing headers can merit review, but their relevance depends on the response and application. **HSTS is meaningful for HTTPS deployments**, so do not treat its absence on local HTTP-only Juice Shop as a confirmed flaw.

In [ ]:
headers_to_check = ["Content-Security-Policy", "X-Content-Type-Options"]

for name in headers_to_check:
    if name in response.headers:
        print(name, "is present")
    else:
        print(name, "is not present; investigate its relevance")

**Practice K:** Look at the `Server` response header using `.get("Server")`. What information, if any, is revealed? Why is the presence of a `Server` header not automatically a critical vulnerability?

### C5. Conditional alerts and avoiding duplicates
`if` statements let a program report findings only when a condition is met. A `set` can remember previously reported alert messages, avoiding repeated warnings. This small example uses sample strings instead of web requests.

In [ ]:
alerts = []
seen = set()

for message in ["Missing CSP", "Missing CSP", "Missing X-Content-Type-Options"]:
    if message not in seen:
        seen.add(message)
        alerts.append(message)

print(alerts)

**Practice L:** Why does the list contain only two alerts even though the loop processes three messages? In your final detector, decide what makes two alerts the “same” finding (for example, category + endpoint + observation).

### C6. Writing observations to a CSV file
CSV files make it easy to include scanner output in a report. `csv.DictWriter` writes a list of dictionaries as rows. The example below saves **practice records**, not final scanner output.

In [ ]:
import csv

practice_records = [
    {"endpoint": "/", "status": 200},
    {"endpoint": "/example", "status": 404}
]

with open("practice_records.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["endpoint", "status"])
    writer.writeheader()
    writer.writerows(practice_records)

print("Saved practice_records.csv")

**Practice M:** Open `practice_records.csv` and identify its column headings and two rows. Why is CSV useful for the final lab report?

# Part D — Your programming assignment (build both scripts yourself)

**Now you have learned the required Python techniques.** Develop the following two programs in separate `.py` files. You may reuse and adapt the short examples above, but **do not copy the entire notebook into a script**. Plan your program, then build and test one requirement at a time.

You may use an editor such as VS Code or create the python files. Both scripts must run from a terminal with `python filename.py`.

## Task 1 — Develop `http_scanner.py` (20 points)

**Purpose:** Collect HTTP observations from known endpoints. This script does **not** decide whether the server is vulnerable.

**Required behavior**
1. Define `BASE_URL = "http://localhost:3000"` and a reasonable timeout.
2. Test **GET** on `/` and GET on `/rest/products/search` using at least two ordinary search terms.
3. Send one **POST** to a selected known path and record the response, even if the method is unsupported.
4. For every request, record: **endpoint, HTTP method, input/payload, status code, response length**, and `Content-Type`. Record the `Server` header when available.
5. Handle connection failures without crashing.
6. Print an easy-to-read record for each request.
7. Save results as `http_scan_results.csv`.

**Suggested development order**
- Start with a single GET request that prints its status.
- Turn its observations into one dictionary.
- Store several test cases in a list and loop through them.
- Add POST and exception handling.
- Finally, save the list of observations to CSV.



## Task 2 — Develop `vulnerability_detector.py` 

**Purpose:** Flag **possible** security indicators for manual review. This is not a penetration-testing framework, and it must not report a confirmed vulnerability based only on a simple automated match.

**Required behavior**
1. Test `/rest/products/search` with a harmless text marker and one non-executable HTML input; check for unchanged reflection in the HTTP response.
2. Compare a normal search term with at least two **simple SQL syntax probes** (for example `'` and `')`). Check for HTTP 500 responses or recognizable database error text.
3. Inspect `Content-Security-Policy` and `X-Content-Type-Options` on the homepage. Record missing headers as **configuration review**, not proof of exploitation.
4. If a `Server` header exists, report it as **informational**, not automatically high severity.
5. Prevent duplicate alerts.
6. Print a concise summary with the **category, endpoint, observation and review status**.
7. Save findings to `vulnerability_findings.csv`.
8. Handle failed requests gracefully. If no indicators are found, explain that this **does not prove the site is secure**.

**Suggested development order**
- Start with a marker reflection check.
- Add a normal-input baseline and one SQL syntax probe.
- Add a small loop over several inputs.
- Add security-header checks.
- Decide how to store and deduplicate findings.
- Print the summary and export CSV.



## Task 3 — Server and configuration analysis 
Use your two scripts' output to explain:
- Which response headers were present and missing, and why the context matters.
- Whether the server reveals potentially useful implementation information.
- Why HTTPS/TLS cannot be meaningfully assessed solely by checking the local HTTP endpoint; describe what you would examine in an actual HTTPS deployment.
- Why a missing header or server error does not automatically mean a vulnerability is exploitable.

## Task 4 — Evidence and reproducibility 
Provide screenshots or captured console output showing:
- Your Juice Shop instance running locally.
- Successful execution of **both** Python scripts.
- The generated CSV files and representative entries.

Include the Python and Juice Shop versions if available, and describe any errors you encountered and resolved.

## Task 5 — Reflection and security analysis 
Write **300–400 words** addressing:
1. What did your scanner observe, and what did your detector flag? Distinguish evidence from assumptions.
2. If you were the security lead for Juice Shop, what would you investigate or fix first, and why?
3. How does this lab extend Lab 4's passive packet analysis into application-layer testing?
4. If the application used HTTPS, what could a network packet capture observe compared with a Python HTTP client making an authorized request?
5. Why should automated findings be manually verified before reporting them as confirmed vulnerabilities?

# Submission checklist and grading
Submit **one PDF report** named `Lab5-FirstName-LastName-StudentNumber.pdf` containing your screenshots for executing `http_scanner.py` and `vulnerability_detector.py`, relevant output, the 300–400-word analysis, and answers to the practice/reflection questions. Also submit both `.py` files separately in your GitHub.

| Component | Points |
|---|---:|
| `http_scanner.py` — requests, logging, CSV, errors | 20 |
| `vulnerability_detector.py` — indicator checks, deduplication, summary | 25 |
| Server and configuration analysis | 15 |
| Evidence and reproducibility | 10 |
| Reflection and security analysis | 30 |
| **Total** | **100** |

**Reminder:** All testing must remain on your own local Juice Shop instance. Do not interpret automated indicators as confirmed vulnerabilities without additional evidence.